# Build an LLM from Scratch

## Tokenising Text

### Loading the Text

In [1]:
import urllib.request
from pathlib import Path

TEXT_FILE_LOC = Path("./the-verdict.txt")
TEXT_URL = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not TEXT_FILE_LOC.exists():
    urllib.request.urlretrieve(TEXT_URL, TEXT_FILE_LOC)

In [2]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    TEXT = f.read()

print(f"Length of text (should be 20479): {len(TEXT)}")

Length of text (should be 20479): 20479


### Generating the Text Corpus

In [3]:
import re

re_split = re.split(r'([,.:;?_!"()\']|--|\s)', TEXT)
words = [item.strip() for item in re_split if item.strip() != ""]

In [4]:
print(f"Number of tokens: {len(words)}")
print("\nExample tokens:")
print(words[:10])

Number of tokens: 4690

Example tokens:
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius']


### Converting Tokens to Token IDs

In [5]:
# Remove duplicates
words = list(sorted(set(words)))
VOCAB_SIZE = len(words)

print(f"Number of tokens: {VOCAB_SIZE}")

Number of tokens: 1130


In [6]:
vocab = {
    token: integer
    for integer, token in enumerate(words)
}

print("Vocab:")

for token, integer in tuple(vocab.items())[20:30]:
    print(f"ID {integer} --> {token}")

Vocab:
ID 20 --> Begin
ID 21 --> Burlington
ID 22 --> But
ID 23 --> By
ID 24 --> Carlo
ID 25 --> Chicago
ID 26 --> Claude
ID 27 --> Come
ID 28 --> Croft
ID 29 --> Destroyed


### Tokenising

In [7]:
class SimpleTokeniserV1:
    def __init__(self, vocab: dict[str, int]) -> None:
        self.str_to_int: dict[str, int] = vocab
        self.int_to_str: dict[int, str] = {i: s for s, i in vocab.items()}

    def encode(self, text: str) -> list[int]:
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip()
            for item in preprocessed
            if item.strip() != ""
        ]
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids: list[int]) -> str:
        text = " ".join([self.int_to_str[i] for i in ids])
        return re.sub(r'\s+([,.?!"()\'])', r'\1', text)

In [8]:
tokeniser = SimpleTokeniserV1(vocab=vocab)
sample_text = "It's the last he painted, you know, Mrs. Gisburn said with pardonable pride."
sample_ids = tokeniser.encode(sample_text)
sample_decode = tokeniser.decode(sample_ids)

print("INPUT:")
print(sample_text)
print()
print("TOKENS:")
print(sample_ids)
print()
print("DECODED TOKENS:")
print(sample_decode)

INPUT:
It's the last he painted, you know, Mrs. Gisburn said with pardonable pride.

TOKENS:
[56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 67, 7, 38, 851, 1108, 754, 793, 7]

DECODED TOKENS:
It' s the last he painted, you know, Mrs. Gisburn said with pardonable pride.


### Adding Special Context Tokens 

In [9]:
try:
    sample_text = "This text will not be convered by the tokeniser!"
    sample_ids = tokeniser.encode(sample_text)
    sample_decode = tokeniser.decode(sample_ids)
except KeyError:
    print("Failed because the vocab does not have these tokens!")

Failed because the vocab does not have these tokens!


In [10]:
words.extend(["<|unk|>", "<|endoftext|>"])

vocab = {
    token: integer
    for integer, token in enumerate(words)
}

In [11]:
class SimpleTokeniserV2:
    def __init__(self, vocab: dict[str, int]) -> None:
        self.str_to_int: dict[str, int] = vocab
        self.int_to_str: dict[int, str] = {i: s for s, i in vocab.items()}

    def encode(self, text: str) -> list[int]:
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip()
            for item in preprocessed
            if item.strip() != ""
        ]
        preprocessed = [
            item
            if item in self.str_to_int else "<|unk|>"
            for item in preprocessed
        ]
        print(preprocessed)
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids: list[int]) -> str:
        text = " ".join([self.int_to_str[i] for i in ids])
        return re.sub(r'\s+([,.?!"()\'])', r'\1', text)

In [12]:
tokeniser = SimpleTokeniserV2(vocab=vocab)
sample_text = "This text will not be convered by the tokeniser!"
sample_ids = tokeniser.encode(sample_text)
sample_decode = tokeniser.decode(sample_ids)

print("INPUT:")
print(sample_text)
print()
print("TOKENS:")
print(sample_ids)
print()
print("DECODED TOKENS:")
print(sample_decode)

['This', '<|unk|>', '<|unk|>', 'not', 'be', '<|unk|>', 'by', 'the', '<|unk|>', '!']
INPUT:
This text will not be convered by the tokeniser!

TOKENS:
[97, 1130, 1130, 711, 198, 1130, 241, 988, 1130, 0]

DECODED TOKENS:
This <|unk|> <|unk|> not be <|unk|> by the <|unk|>!
